In [ ]:
import pandas as pd

url = "/Users/suheylatozan/Desktop/Movement Recovery Lab/sc_ramp.csv"
df = pd.read_csv(url)

In [71]:
df.shape

(1295, 38)

In [72]:
idx = (df.participant == "s109") & (df.recr_curve == "scramp-002")
temp_df = df[idx].reset_index(drop=True).copy()

In [73]:
import seaborn as sns

# sns.scatterplot(x=temp_df["sc_current"], y=temp_df["FCR"])

In [74]:
df

,mode,recr_curve,sc_current,cx_voltage,datalength,sweep,timestamp,timestamp_s,timestamp_ms,Fs,...,LFCR,LAPB,LADM,RDeltoid,ssep_size,hue_ADM,hue_APB,hue_ECR,hue_FCR,hue_Triceps
0,research_paired_repeat,scramp-001,0.000000,72.064078,901,0.15,63136155930000,63136155.93,63136155930,6006.666667,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
1,research_paired_repeat,scramp-001,0.000000,73.240733,901,0.15,63136157930000,63136157.93,63136157930,6006.666667,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2,research_paired_repeat,scramp-001,0.000000,72.946569,901,0.15,63136159930000,63136159.93,63136159930,6006.666667,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
3,research_paired_repeat,scramp-001,0.000000,72.064274,901,0.15,63136167350000,63136167.35,63136167350,6006.666667,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
4,research_paired_repeat,scramp-001,0.000000,72.946374,901,0.15,63136169350000,63136169.35,63136169350,6006.666667,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1290,research_paired_repeat,scramp-006,4.731587,51.207570,901,0.15,63141690920000,63141690.92,63141690920,6006.666667,...,NaN,NaN,NaN,243.677947,901.0,0,0,0,0,0
1291,research_paired_repeat,scramp-006,4.731587,51.207570,901,0.15,63141692920000,63141692.92,63141692920,6006.666667,...,NaN,NaN,NaN,254.469659,901.0,0,0,0,0,0
1292,research_paired_repeat,scramp-006,4.731587,51.207570,901,0.15,63141694920000,63141694.92,63141694920,6006.666667,...,NaN,NaN,NaN,223.144381,901.0,0,0,0,0,0
1293,research_paired_repeat,scramp-006,4.731587,51.207570,901,0.15,63141696920000,63141696.92,63141696920,6006.666667,...,NaN,NaN,NaN,284.652321,901.0,0,0,0,0,0


In [75]:
import os

current_directory = os.getcwd()
output_path = os.path.join(current_directory, "paired_dataset.pdf")

# Plot dataset and save it as a PDF
# model.plot(df, output_path=output_path)

In [76]:
# ============================================
# Imports
# ============================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import jax
import jax.numpy as jnp
import numpyro as pyro
import numpyro.distributions as dist

# Disable JAX warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")


In [77]:
import logging

import numpy as np
import jax
import jax.numpy as jnp
import numpyro as pyro
import numpyro.distributions as dist

from hbmep import functional as F, smooth_functional as SF
from hbmep.model import BaseModel
from hbmep.util import site

EPS = 1e-3


class HB(BaseModel):
    def __init__(self, *args, **kw):
        super(HB, self).__init__(*args, **kw)
        self.use_mixture = False

    def hb_rl(self, intensity, features, response=None, **kw):
        num_data = intensity.shape[0]
        num_features = np.max(features, axis=0) + 1

        # Mask missing observations
        mask_obs = True
        if response is not None:
            mask_obs = np.invert(np.isnan(response))

        # Hyper-priors
        a_loc = pyro.sample(
            site.a.log, dist.TruncatedNormal(5., 10., low=0)
        )
        a_scale = pyro.sample(site.a.scale, dist.HalfNormal(10.))
        b_scale = pyro.sample(site.b.scale, dist.HalfNormal(10.))
        h_scale = pyro.sample(site.h.scale, dist.HalfNormal(50.))

        g_scale = pyro.sample(site.g.scale, dist.HalfNormal(5.))
        v_scale = pyro.sample(site.v.scale, dist.HalfNormal(1.))

        c1_scale = pyro.sample(site.c1.scale, dist.HalfNormal(5.))
        c2_scale = pyro.sample(site.c2.scale, dist.HalfNormal(.5))

        # Priors
        with pyro.plate(site.num_response, self.num_response):
            with pyro.plate_stack(
                site.num_features, num_features, rightmost_dim=-2
            ):
                a = pyro.sample(
                    site.a, dist.TruncatedNormal(a_loc, a_scale, low=0)
                )

                b_raw = pyro.sample(site.b.raw, dist.HalfNormal(1))
                b = pyro.deterministic(site.b, b_scale * b_raw)

                g_raw = pyro.sample(site.g.raw, dist.HalfNormal(1))
                g = pyro.deterministic(site.g, g_scale * g_raw)

                h_raw = pyro.sample(site.h.raw, dist.HalfNormal(1))
                h = pyro.deterministic(site.h, h_scale * h_raw)

                v_raw = pyro.sample(site.v.raw, dist.HalfNormal(1))
                v = pyro.deterministic(site.v, v_scale * v_raw)

                c1_raw = pyro.sample(site.c1.raw, dist.HalfNormal(1))
                c1 = pyro.deterministic(site.c1, c1_scale * c1_raw)

                c2_raw = pyro.sample(site.c2.raw, dist.HalfNormal(1))
                c2 = pyro.deterministic(site.c2, c2_scale * c2_raw)

        # Outlier probability
        if self.use_mixture:
            q = pyro.sample(site.outlier_prob, dist.Uniform(0., 0.01))

        # Observation model
        with pyro.handlers.mask(mask=mask_obs):
            with pyro.plate(site.num_response, self.num_response):
                with pyro.plate(site.num_data, num_data):
                    mu = SF.rectified_logistic(
                        intensity,
                        a[*features.T],
                        b[*features.T],
                        g[*features.T],
                        h[*features.T],
                        v[*features.T],
                        EPS
                    )
                    alpha, beta = self.gamma_likelihood(
                        mu, 
                        c1[*features.T],
                        c2[*features.T],
                    )
                    pyro.deterministic(site.mu, mu)

                    # Mixture distribution
                    if self.use_mixture:
                        mixing_distribution = dist.Categorical(
                            probs=jnp.stack([1 - q, q], axis=-1)
                        )
                        component_distributions=[
                            dist.Gamma(concentration=alpha, rate=beta),
                            dist.HalfNormal(
                                scale=(g[*features.T] + h[*features.T])
                            )
                        ]
                        Mixture = dist.MixtureGeneral(
                            mixing_distribution=mixing_distribution,
                            component_distributions=component_distributions
                        )

                    # Observations
                    y_ = pyro.sample(
                        site.obs,
                        (
                            Mixture if self.use_mixture
                            else dist.Gamma(concentration=alpha, rate=beta)
                        ),
                        obs=response
                    )


In [78]:
# ============================================
# Least Squares Recruitment Curve Optimization
# ============================================

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def rectified_logistic(x01, a, b, g, h, v):
    # v controls sharpness
    # use np.maximum instead of Python max() for vectorization
    return g + np.maximum(0, (-v + (h + v) * sigmoid(b * (x01 - a) - np.log(h / v))))

def scale_minmax(x):
    xmin, xmax = float(np.min(x)), float(np.max(x))
    rng = xmax - xmin if xmax > xmin else 1.0
    return (x - xmin) / rng, xmin, rng

def fit_rectified_logistic(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if x.size < 5:
        raise ValueError("Need at least 5 points")
    x01, xmin, xrng = scale_minmax(x)
    g0 = np.percentile(y, 10.0)
    u0 = np.percentile(y, 90.0)
    h0 = max(u0 - g0, 1e-6)

    # Smart midpoint initialization for a0
    target = g0 + 0.5 * h0
    idx = np.argmin(np.abs(y - target))
    a0 = x01[idx]
    b0, v0 = 5.0, 1.0

    p0 = np.array([a0, b0, g0, h0, v0], float) # Try to reinitialize - parallelization, sample from lower to upper bound. 
    lb = np.array([1e-4, 1e-4, 1e-4, 1e-4, 1e-4])
    ub = np.array([20.0, 500.0, 10.0, 2000.0, 50.0])

    def residuals(theta):
        a, b, g, h, v = theta 
        mu = rectified_logistic(x01, a, b, g, h, v)
        return mu - y

    try:
        res = least_squares(residuals, p0, bounds=(lb, ub),
                            loss="soft_l1", f_scale=1.0, max_nfev=30000)
        success = res.success and np.isfinite(res.x).all()
        a, b, g, h, v = res.x
        mu = rectified_logistic(x01, a, b, g, h, v)
        rmse = float(np.sqrt(np.mean((y - mu) ** 2)))
        ss_res = np.sum((y - mu) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    except Exception:
        success = False
        a = b = g = h = v = np.nan
        rmse = r2 = np.nan

    return {
        "params": dict(a=a, b=b, g=g, h=h, v=v, xmin=xmin, xrng=xrng),
        "rmse": rmse,
        "r2": r2,
        "success": bool(success),
    }

def predict_ls_curve(x_new, fit):
    p = fit["params"]
    x01 = (np.asarray(x_new) - p["xmin"]) / (p["xrng"] if p["xrng"] != 0 else 1.0)
    return rectified_logistic(x01, p["a"], p["b"], p["g"], p["h"], p["v"])



In [79]:
csv_path = "/Users/suheylatozan/Desktop/Movement Recovery Lab/sc_ramp.csv"
df = pd.read_csv(csv_path)
req_cols = ["sc_current", "FCR", "participant", "recr_curve"]
df = df.dropna(subset=req_cols).copy()

group_cols = ["participant", "recr_curve"]


In [80]:
#Run hbMEP Bayesian Model

model = HB()
model.intensity = "sc_current"
model.features = group_cols
model.response = ["FCR"]
model._model = model.hb_rl
model.use_mixture = True
model.mcmc_params = dict(num_chains=2, num_warmup=500, num_samples=250) # Only way to determine is by analyzing convergence diagnostics. 

# Encode and run
df_enc, enc = model.load(df)
mcmc, posterior = model.run(df=df_enc)
pred_df = model.make_prediction_dataset(df=df_enc, num_points=200)
predictive = model.predict(df=pred_df, posterior=posterior, num_samples=500, return_sites=[site.mu])
pred_df["mu_post_mean"] = np.asarray(predictive[site.mu]).mean(axis=0)


Running chain 0: 100%|██████████| 750/750 [08:57<00:00,  1.40it/s]


In [91]:
mcmc.print_summary()


                   mean       std    median      5.0%     95.0%     n_eff     r_hat
     a[0,0,0]      0.47      0.12      0.48      0.28      0.64    196.11      1.01
     a[0,1,0]      2.13      0.12      2.16      2.05      2.31     90.87      1.01
     a[0,2,0]      0.36      0.16      0.36      0.06      0.58    538.01      1.00
     a[0,3,0]      1.21      0.15      1.22      1.02      1.44    168.54      1.01
     a[0,4,0]      1.68      0.25      1.69      1.28      1.98     76.14      1.04
     a[0,5,0]      1.97      0.28      2.08      1.40      2.21    133.99      1.01
     a[0,6,0]      0.61      0.20      0.66      0.30      0.96    130.79      1.00
     a[0,7,0]      0.53      0.05      0.53      0.45      0.62    464.37      1.00
     a[0,8,0]      0.79      0.22      0.82      0.35      1.06    235.54      1.00
     a[1,0,0]      1.13      0.14      1.15      0.99      1.31     65.63      1.01
     a[1,1,0]      0.83      0.09      0.83      0.69      0.97    489.11  

In [81]:
# --- Decode categorical features using LabelEncoder objects ---
for col in model.features:
    if col in pred_df and col in enc:
        le = enc[col]  # this is a LabelEncoder
        if hasattr(le, "classes_"):
            pred_df[col] = le.inverse_transform(pred_df[col].astype(int))

print("Decoded participants in pred_df:", pred_df["participant"].unique())
print("Decoded recr_curve in pred_df:", pred_df["recr_curve"].unique())


Decoded participants in pred_df: ['s101' 's109' 's114' 's118']
Decoded recr_curve in pred_df: ['scramp-001' 'scramp-002' 'scramp-003' 'scramp-004' 'scramp-005'
 'scramp-006' 'scramp-007' 'scramp-008' 'scramp-009']


In [82]:
# ============================================
# Helper: Fit least-squares curve for each participant/recruitment curve group
# ============================================

def fit_by_group_ls(df, intensity_col, response_col, group_cols):
    rows = []
    for combo, gdf in df.groupby(group_cols):
        combo = combo if isinstance(combo, tuple) else (combo,)
        try:
            fit = fit_rectified_logistic(
                gdf[intensity_col].values,
                gdf[response_col].values
            )
            row = {c: v for c, v in zip(group_cols, combo)}
            row.update(fit["params"])
            row.update(dict(
                success=fit["success"],
                rmse=fit["rmse"],
                r2=fit["r2"]
            ))
            rows.append(row)
        except Exception as e:
            # Handles cases where the group doesn't have enough data or fails to fit
            row = {c: v for c, v in zip(group_cols, combo)}
            row.update(dict(
                a=np.nan, b=np.nan, g=np.nan, h=np.nan,
                xmin=np.nan, xrng=np.nan, success=False,
                rmse=np.nan, r2=np.nan
            ))
            rows.append(row)
    return pd.DataFrame(rows)


In [83]:
# ============================================
# 3) Fit Least Squares Curves per Group
# ============================================

group_cols = ["participant", "recr_curve"]
ls_table = fit_by_group_ls(df, intensity_col="sc_current", response_col="FCR", group_cols=group_cols)
ls_table.head()


,participant,recr_curve,a,b,g,h,v,xmin,xrng,success,rmse,r2
0,s101,scramp-001,0.187791,9.213171,0.563902,24.310392,3.463932,0.0,5.049677,True,1.841031,0.965310
1,s101,scramp-002,0.405791,18.079574,0.365371,2.781106,3.683734,0.0,5.427408,True,0.301739,0.950052
2,s101,scramp-003,0.048602,11.737342,2.101895,44.973555,0.932798,0.0,5.447289,True,2.938552,0.975467
3,s101,scramp-004,0.251994,6.427164,0.374545,7.531893,50.000000,0.0,5.447289,True,1.625887,0.792512
4,s101,scramp-005,0.417055,26.355337,0.355035,3.354762,6.286125,0.0,5.427408,True,0.287012,0.969830


In [84]:
print(df["participant"].unique())
print(pred_df["participant"].unique())


['s101' 's109' 's114' 's118']
['s101' 's109' 's114' 's118']


In [85]:
def compare_and_plot(
    df, pred_df, ls_table,
    intensity_col="sc_current", response_col="FCR",
    group_cols=("participant", "recr_curve"),
    disagreement_threshold=0.25
):
    reports = []
    unique_groups = list(ls_table[list(group_cols)].drop_duplicates().itertuples(index=False, name=None))
    n = len(unique_groups)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 4.5 * nrows), squeeze=False)

    for idx, grp in enumerate(unique_groups):
        ax = axes[idx // ncols][idx % ncols]
        mask_data = np.ones(len(df), dtype=bool)
        mask_pred = np.ones(len(pred_df), dtype=bool)
        for c, v in zip(group_cols, grp):
            mask_data &= (df[c] == v)
            mask_pred &= (pred_df[c] == v)

        gdf = df.loc[mask_data, [intensity_col, response_col]].sort_values(intensity_col)
        gpred = pred_df.loc[mask_pred, [intensity_col, "mu_post_mean"]].sort_values(intensity_col)

        # Always plot data
        ax.scatter(gdf[intensity_col], gdf[response_col], s=14, alpha=0.65, color='k', label="Data")

        if gpred.empty:
            ax.set_title(f"{grp} - No pred_df match")
            continue

        row = ls_table
        for c, v in zip(group_cols, grp):
            row = row[row[c] == v]

        if len(row) == 1 and bool(row.iloc[0]["success"]):
            fit_params = {"params": {k: row.iloc[0][k] for k in ["a", "b", "g", "h", "v", "xmin", "xrng"]}}
            xgrid = np.linspace(gdf[intensity_col].min(), gdf[intensity_col].max(), 200)
            yhat_ls = predict_ls_curve(xgrid, fit_params)
            xpred = gpred[intensity_col].values
            ypred_mean = gpred["mu_post_mean"].values
            yhat_post = np.interp(xgrid, xpred, ypred_mean)

            denom = max(1e-6, np.percentile(np.abs(yhat_ls), 90.0))
            mad = float(np.mean(np.abs(yhat_ls - yhat_post)) / denom)
            flag = mad > disagreement_threshold

            ax.plot(xgrid, yhat_ls, lw=2, color='r', label="Least squares")
            ax.plot(xgrid, yhat_post, lw=2, linestyle="--", color='b', label="hbMEP mean")
            ax.set_title(f"{grp}\nMAD={mad:.3f} {'FLAG' if flag else ''}")
            ax.legend()
            reports.append({**{c: v for c, v in zip(group_cols, grp)}, "mad": mad, "flag": flag})
        else:
            # Fallback: mean line or message
            ax.set_title(f"{grp} - LS fit failed (showing raw data)")
            if len(gdf) > 1:
                ax.plot([gdf[intensity_col].min(), gdf[intensity_col].max()],
                        [np.mean(gdf[response_col])] * 2,
                        linestyle='--', color='gray', label="Mean reference")
            ax.legend()

    plt.tight_layout()
    report_df = pd.DataFrame(reports)
    return fig, report_df



In [86]:
fig, report = compare_and_plot(df, pred_df, ls_table)

# Ensure output folder exists
import os
from hbmep.util import make_pdf

if not getattr(model, "build_dir", None) or not model.build_dir:
    model.build_dir = os.path.join(os.getcwd(), "hbmep_outputs")

os.makedirs(model.build_dir, exist_ok=True)
output_path = os.path.join(model.build_dir, "paired_curves_optimized_test.pdf")

# Save figure directly
make_pdf(figures=[fig], output_path=output_path)
print(f"Saved paired comparison PDF to: {output_path}")





Saved paired comparison PDF to: /Users/suheylatozan/Desktop/Movement Recovery Lab/hbmep/notebooks/hbmep_outputs/paired_curves_optimized_test.pdf


In [87]:
import os
print(os.getcwd())


/Users/suheylatozan/Desktop/Movement Recovery Lab/hbmep/notebooks


In [88]:
os.listdir(model.build_dir)



['paired_curves_optimized_test.pdf',
 'paired_curves_optimized_2.1.pdf',
 'paired_curves_optimized_2.2.pdf',
 'paired_curves_optimized_2.pdf']